# LeRover CNN Policy — Colab Training

Trains `LoopPolicyNet` on leorover CNN episodes stored in Google Drive.

**Before running:** set *Runtime → Change runtime type → GPU* (T4 is fine).

| Section | What it does |
|---|---|
| 0 | Verify GPU |
| 1 | Mount Google Drive |
| 2 | Get the code |
| 3 | Install dependencies |
| 4 | Configure paths & hyperparameters |
| 5 | Run training |
| 6 | Plot results |
| 7 | Save checkpoint back to Drive |

## 0 · Verify GPU

In [ ]:
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU detected — go to Runtime → Change runtime type → GPU')

import torch
print(f'\nPyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPU ok  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## 2 · Get the Code

**Option A** — clone from GitHub (set your repo URL below).  
**Option B** — copy a zip from Drive (upload `leorovervla.zip` to your Drive root first).

Set `CODE_SOURCE = 'github'` or `'drive'`.

In [ ]:
import os

CODE_SOURCE  = 'drive'          # 'github' | 'drive'
GITHUB_URL   = 'https://github.com/YOUR_USERNAME/leorovervla.git'
DRIVE_ZIP    = '/content/drive/MyDrive/leorovervla.zip'
PROJECT_ROOT = '/content/leorovervla'

if not os.path.isdir(PROJECT_ROOT):
    if CODE_SOURCE == 'github':
        !git clone {GITHUB_URL} {PROJECT_ROOT}
    else:
        !unzip -q "{DRIVE_ZIP}" -d /content/
        # Rename if the zip extracts a differently named folder
        extracted = [d for d in os.listdir('/content/') if d.startswith('leorovervla')]
        if extracted and extracted[0] != 'leorovervla':
            os.rename(f'/content/{extracted[0]}', PROJECT_ROOT)
else:
    print('Project already present, skipping download.')

%cd {PROJECT_ROOT}
print('Working directory:', os.getcwd())

## 3 · Install Dependencies

In [ ]:
# Colab ships with a recent PyTorch; only extra packages are needed.
!pip install av>=12.0 pandas>=2.2 pyarrow>=15.0 tqdm>=4.66 Pillow>=10.0 --quiet

import torch
assert torch.cuda.is_available(), 'CUDA not available — check runtime type!'
print(f'✓ torch {torch.__version__} | CUDA {torch.version.cuda} | {torch.cuda.get_device_name(0)}')

## 4 · Configure Paths & Hyperparameters

Edit the variables in this cell before running training.

**Drive layout expected:**
```
MyDrive/
  leorover_cnn/
    episodes/
      session_YYYYMMDD_HHMMSS/
        episode_000/
          video.mp4
          data.parquet
          episode_info.json
        ...
```

Set `RESUME_CHECKPOINT` to a `.pt` path to warm-start weights, or leave as `None` to train from scratch.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────
EPISODES_DIR       = '/content/drive/MyDrive/leorover_cnn/episodes'
RUN_DIR            = '/content/drive/MyDrive/leorover_cnn/runs/cnn_v2'
RESUME_CHECKPOINT  = '/content/drive/MyDrive/leorover_cnn/runs/cnn_v2/checkpoints/best.pt'  # or None

# ── Hyperparameters ────────────────────────────────────────────────────
EPOCHS        = 40
BATCH_SIZE    = 64
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 2
VAL_RATIO     = 0.2
FRAME_HISTORY = 3
IMAGE_WIDTH   = 160
IMAGE_HEIGHT  = 120
HUBER_DELTA   = 1.0
SEED          = 42

# ── Sanity-check ───────────────────────────────────────────────────────
import os
assert os.path.isdir(EPISODES_DIR), f'Episodes directory not found: {EPISODES_DIR}'
sessions = [d for d in os.listdir(EPISODES_DIR) if os.path.isdir(os.path.join(EPISODES_DIR, d))]
print(f'✓ Found {len(sessions)} session(s) in {EPISODES_DIR}')
for s in sorted(sessions):
    print(f'  {s}')

if RESUME_CHECKPOINT:
    assert os.path.isfile(RESUME_CHECKPOINT), f'Resume checkpoint not found: {RESUME_CHECKPOINT}'
    print(f'✓ Resume checkpoint: {RESUME_CHECKPOINT}')
else:
    print('Training from scratch (no resume checkpoint).')

## 5 · Run Training

In [ ]:
cmd_parts = [
    'python -m loop_cnn.train',
    f'--episodes-dir  "{EPISODES_DIR}"',
    f'--run-dir       "{RUN_DIR}"',
    f'--epochs        {EPOCHS}',
    f'--batch-size    {BATCH_SIZE}',
    f'--lr            {LR}',
    f'--weight-decay  {WEIGHT_DECAY}',
    f'--num-workers   {NUM_WORKERS}',
    f'--val-ratio     {VAL_RATIO}',
    f'--frame-history {FRAME_HISTORY}',
    f'--image-width   {IMAGE_WIDTH}',
    f'--image-height  {IMAGE_HEIGHT}',
    f'--huber-delta   {HUBER_DELTA}',
    f'--seed          {SEED}',
    '--device cuda',
]
if RESUME_CHECKPOINT:
    cmd_parts.append(f'--resume "{RESUME_CHECKPOINT}"')

cmd = ' '.join(cmd_parts)
print('Command:', cmd)
!{cmd}

## 6 · Plot Results

In [ ]:
import json, glob
import matplotlib.pyplot as plt

# Find the most recent run summary
summaries = sorted(glob.glob(os.path.join(RUN_DIR, '**/training_summary.json'), recursive=True))
assert summaries, f'No training_summary.json found under {RUN_DIR}'
summary_path = summaries[-1]
print(f'Loading: {summary_path}')

with open(summary_path) as f:
    summary = json.load(f)

history = summary['history']
epochs      = [r['epoch']      for r in history]
train_loss  = [r['train_loss'] for r in history]
val_loss    = [r['val_loss']   for r in history]
val_mae_vx  = [r['val_mae_vx']    for r in history]
val_mae_vy  = [r['val_mae_vy']    for r in history]
val_mae_om  = [r['val_mae_omega'] for r in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_loss, label='train loss')
axes[0].plot(epochs, val_loss,   label='val loss')
axes[0].axvline(summary['best_epoch'], color='gray', linestyle='--', label=f'best epoch {summary["best_epoch"]}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Huber Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, val_mae_vx, label='vx')
axes[1].plot(epochs, val_mae_vy, label='vy')
axes[1].plot(epochs, val_mae_om, label='ω')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (normalised)')
axes[1].set_title('Validation MAE per action dim')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Best epoch: {summary["best_epoch"]}  |  best val loss: {summary["best_metric"]:.4f}', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(summary_path), 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Completed {summary["epochs_completed"]}/{summary["epochs_requested"]} epochs')
print(f'Best epoch: {summary["best_epoch"]}  |  best val loss: {summary["best_metric"]:.4f}')

## 7 · Save Checkpoint to Drive

The training already writes checkpoints directly to `RUN_DIR` on Drive.  
This cell just prints where they are for confirmation.

In [ ]:
import glob as _glob

best_ckpts = sorted(_glob.glob(os.path.join(RUN_DIR, '**/best.pt'), recursive=True))
last_ckpts = sorted(_glob.glob(os.path.join(RUN_DIR, '**/last.pt'), recursive=True))

print('Best checkpoints saved to Drive:')
for p in best_ckpts:
    size_mb = os.path.getsize(p) / 1e6
    print(f'  {p}  ({size_mb:.2f} MB)')

print('\nTo load for inference:')
if best_ckpts:
    print(f"  from loop_cnn.model import load_checkpoint")
    print(f"  model, payload = load_checkpoint('{best_ckpts[-1]}')")